<a href="https://colab.research.google.com/github/Poinbreak/ocr_and_formatting/blob/main/Kaggle_OCR_Multi_Model_Testing_Suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================
KAGGLE CELL 1: Environment Setup & Installations
Run this cell first to install all necessary packages for the models.
================================================================================
"""
# Run this in Kaggle (uncomment the line below for the actual notebook cell):
# !pip install torch torchvision transformers accelerate pillow bitsandbytes paddleocr paddlepaddle-gpu chandra-ocr qwen-vl-utils vllm timm

import torch
import gc
from PIL import Image
from transformers import (
    AutoModel, AutoTokenizer, AutoProcessor,
    AutoModelForCausalLM, AutoModelForImageTextToText
)

# A helper function to prevent Kaggle from crashing due to out-of-memory (OOM) errors.
def clear_vram():
    """Wipes the GPU memory cleanly between model tests."""
    gc.collect()
    torch.cuda.empty_cache()
    print("VRAM Cleared.")

# Load your test image (Upload 'test_log.jpg' to your Kaggle Input/Working directory)
TEST_IMAGE_PATH = "test_log.jpg" # Change this to your actual uploaded image name
TEST_PROMPT = "Extract all handwritten text verbatim and format it."

In [ ]:

"""
================================================================================
KAGGLE CELL 2: GOT-OCR 2.0
Highly compressed, stepfun-ai model. Great for coordinate-based OCR.
================================================================================
"""
def test_got_ocr(image_path):
    print("\n--- Loading GOT-OCR 2.0 ---")
    tokenizer = AutoTokenizer.from_pretrained('ucaslcl/GOT-OCR2_0', trust_remote_code=True)
    model = AutoModel.from_pretrained(
        'ucaslcl/GOT-OCR2_0',
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map='cuda',
        use_safetensors=True
    ).eval()

    res = model.chat(tokenizer, image_path, ocr_type='ocr')
    print("GOT-OCR Output:\n", res)

    # Clean up
    del model, tokenizer
    clear_vram()

In [ ]:
"""
================================================================================
KAGGLE CELL 3: GLM-OCR (0.9B)
Zhipu AI's structured layout model. Excellent at LaTeX/Markdown extraction.
================================================================================
"""
def test_glm_ocr(image_path):
    print("\n--- Loading GLM-OCR ---")
    tokenizer = AutoTokenizer.from_pretrained('zai-org/GLM-OCR', trust_remote_code=True)
    model = AutoModel.from_pretrained(
        'zai-org/GLM-OCR',
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map='cuda'
    ).eval()

    # GLM-OCR specific chatting format
    res, _ = model.chat(tokenizer, image_path, TEST_PROMPT)
    print("GLM-OCR Output:\n", res)

    del model, tokenizer
    clear_vram()


In [ ]:
"""
================================================================================
KAGGLE CELL 4: PaddleOCR
Traditional, incredibly fast pipeline detector. Great for edge CPUs.
================================================================================
"""
def test_paddleocr(image_path):
    print("\n--- Running PaddleOCR ---")
    from paddleocr import PaddleOCR

    # Loads the lightweight detection/recognition models into GPU automatically
    ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=True)
    result = ocr.ocr(image_path, cls=True)

    print("PaddleOCR Output:")
    for idx in range(len(result)):
        res = result[idx]
        for line in res:
            print(f"Text: {line[1][0]} | Confidence: {line[1][1]:.2f}")

    # Paddle handles its own C++ memory, but we can clean python references
    del ocr
    clear_vram()

In [ ]:
"""
================================================================================
KAGGLE CELL 5: The Qwen Series (Qwen2.5-VL-3B, Qwen3-8B, Qwen3.5-0.8B)
Alibaba's SOTA models. We load them in 4-bit/8-bit to fit the Kaggle T4.
================================================================================
"""
def test_qwen_family(image_path, model_id="Qwen/Qwen2.5-VL-3B-Instruct"):
    print(f"\n--- Loading {model_id} ---")
    from qwen_vl_utils import process_vision_info

    # Load with bfloat16 or 4-bit quantization if it's the 8B model
    load_kwargs = {"device_map": "auto", "torch_dtype": torch.bfloat16}
    if "8B" in model_id:
        load_kwargs["load_in_4bit"] = True # Force 4-bit to fit 16GB Kaggle limit

    model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, **load_kwargs)
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": TEST_PROMPT},
            ],
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    print(f"{model_id} Output:\n", output_text[0])

    del model, processor, inputs
    clear_vram()


In [ ]:
"""
================================================================================
KAGGLE CELL 6: Chandra 2 & LocateAnything
Next-generation 2026 models for layout preservation and zero-shot grounding.
================================================================================
"""
def test_chandra_2(image_path):
    print("\n--- Loading Chandra OCR 2 ---")
    # Chandra 2 runs on top of Qwen3.5, explicitly designed for Markdown/JSON extraction
    model = AutoModelForImageTextToText.from_pretrained(
        "datalab-to/chandra-ocr-2",
        device_map="auto",
        torch_dtype=torch.bfloat16
    ).eval()
    processor = AutoProcessor.from_pretrained("datalab-to/chandra-ocr-2")
    processor.tokenizer.padding_side = "left"

    # Quick HuggingFace inference (Using Chandra's internal batch structure conceptually)
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, text="Extract to JSON", return_tensors="pt").to("cuda", torch.bfloat16)

    outputs = model.generate(**inputs, max_new_tokens=512)
    print("Chandra 2 Output:\n", processor.decode(outputs[0], skip_special_tokens=True))

    del model, processor, inputs
    clear_vram()

def test_locate_anything(image_path, target_object="handwritten notes"):
    print("\n--- Loading NVIDIA LocateAnything ---")
    # Parallel Box Decoding model for precise visual grounding
    model = AutoModel.from_pretrained("nvidia/LocateAnything", trust_remote_code=True, device_map="auto").eval()

    # Note: LocateAnything API usage differs slightly by version, standard multimodal prompt shown:
    image = Image.open(image_path).convert("RGB")
    # It returns bounding box coordinates natively based on the prompt
    res = model.chat(image=image, prompt=f"Locate: {target_object}")
    print(f"LocateAnything Grounding for '{target_object}':\n", res)

    del model
    clear_vram()

In [ ]:
"""
================================================================================
KAGGLE CELL 7: PaliGemma (Google) & MiniCPM-V 2.6 (OpenBMB)
Heavyweight VLMs for complex visual comprehension.
================================================================================
"""
def test_paligemma(image_path):
    print("\n--- Loading PaliGemma 3B ---")
    from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor

    model_id = "google/paligemma-3b-pt-224"
    model = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="cuda").eval()
    processor = PaliGemmaProcessor.from_pretrained(model_id)

    image = Image.open(image_path).convert("RGB")
    inputs = processor(text=TEST_PROMPT, images=image, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        generation = model.generate(**inputs, max_new_tokens=256)
        # Slicing the input prompt off the generated output
        generation = generation[0][inputs["input_ids"].shape[-1]:]
        decoded = processor.decode(generation, skip_special_tokens=True)

    print("PaliGemma Output:\n", decoded)
    del model, processor, inputs
    clear_vram()

def test_minicpm(image_path):
    print("\n--- Loading MiniCPM-V 2.6 ---")
    model = AutoModel.from_pretrained(
        'openbmb/MiniCPM-V-2_6',
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto"
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-V-2_6', trust_remote_code=True)

    image = Image.open(image_path).convert('RGB')
    msgs = [{'role': 'user', 'content': [image, TEST_PROMPT]}]

    res = model.chat(image=None, msgs=msgs, tokenizer=tokenizer)
    print("MiniCPM Output:\n", res)

    del model, tokenizer
    clear_vram()


In [ ]:
"""
================================================================================
KAGGLE CELL 8: Execute the Tests!
Run these ONE AT A TIME to prevent running out of GPU VRAM.
================================================================================
"""
if __name__ == "__main__":
    # Upload an image to Kaggle named 'test_log.jpg' before running these.

    # test_got_ocr(TEST_IMAGE_PATH)
    # test_glm_ocr(TEST_IMAGE_PATH)
    # test_paddleocr(TEST_IMAGE_PATH)

    # Qwen Family testing:
    # test_qwen_family(TEST_IMAGE_PATH, "Qwen/Qwen2.5-VL-3B-Instruct")
    # test_qwen_family(TEST_IMAGE_PATH, "Qwen/Qwen3.5-0.8B") # Super fast edge model
    # test_qwen_family(TEST_IMAGE_PATH, "Qwen/Qwen3-8B")     # Note: Loaded in 4-bit automatically

    # 2026 Layout/Grounding Models
    # test_chandra_2(TEST_IMAGE_PATH)
    # test_locate_anything(TEST_IMAGE_PATH, "signatures and tables")

    # Heavyweights
    # test_paligemma(TEST_IMAGE_PATH)
    # test_minicpm(TEST_IMAGE_PATH)
    pass